# Pydantic (richest option — validation + field descriptions)

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model(
    model="openai/gpt-oss-120b",
    model_provider="groq",
    temperature=0.7
)

What it is: A library (pydantic) that provides data validation and parsing using Python type hints.

How it works: You define a class that inherits from BaseModel. Pydantic automatically validates input data against the declared types and can coerce types (e.g., "123" → int).

Use case: Perfect for APIs, structured outputs from LLMs, or anywhere you need validation + serialization.

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    rating: float = Field(description="The movies rating out of 10")

model_with_structure = model.with_structured_output(Movie)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

## Nested Pydantic models

In [9]:
class Actor(BaseModel):
    name: str
    role: str
class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Hulk")
response

MovieDetails(title='Hulk', year=2003, cast=[Actor(name='Eric Bana', role='Bruce Banner / Hulk'), Actor(name='Jennifer Connelly', role='Betty Ross'), Actor(name='Sam Elliot', role='David Banner'), Actor(name='Liv Tyler', role='Ellie'), Actor(name='William Hurt', role='General Ross')], genres=['Action', 'Adventure', 'Sci-Fi'], budget=138000000.0)

# TypedDict (lightweight, no runtime validation)

What it is: A way to define the expected keys and value types of a dictionary.

How it works: It’s purely for type checking (static analysis with tools like mypy), not runtime validation.

Use case: When you want dictionaries with fixed keys but don’t need runtime validation.

Annotated (from typing_extensions)
What it is: A way to attach extra metadata to type hints.

How it works: Wraps a type with annotations (e.g., validation rules, descriptions).

Use case: Useful with frameworks (like FastAPI or Pydantic v2) to add constraints or documentation.

In [12]:
from typing_extensions import TypedDict, Annotated
class MovieDict(TypedDict):
    """A movie with details."""
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withtypedict = model.with_structured_output(MovieDict)
response = model_withtypedict.invoke("Please provide the details of the movie avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

# Structured output on create_agent (dataclasses too)

In [31]:
from pydantic import BaseModel, Field
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq

# Define structured schema
class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")

# Initialize Groq model
model = ChatGroq(
    model="openai/gpt-oss-120b",  # free Groq model
    temperature=0.7,
)

# Wrap with structured output
structured_model = model.with_structured_output(ContactInfo)

# Invoke with user message
result = structured_model.invoke([
    HumanMessage(content="Extract contact info from: John Doe, john@example.com, (555)123-4567")
])

print(result)


name='John Doe' email='john@example.com' phone='(555)123-4567'


| Feature | ``TypedDict`` | ``Annotated`` | ``pydantic.BaseModel`` |
| --- | --- | --- | --- |
| Runtime validation | ❌ No | ❌ No (metadata only) | ✅ Yes |
| Type hints | ✅ Yes | ✅ Yes | ✅ Yes |
| Metadata (docs, etc) | ❌ No | ✅ Yes | ✅ Yes (via ``Field``) |
| Serialization | ❌ No | ❌ No | ✅ Yes (``dict()``, ``json()``) |
| Best for | Static typing | Adding constraints | Full validation + parsing |

👉 In short:

TypedDict → static typing for dicts.

Annotated → add metadata/constraints to types.

BaseModel (Pydantic) → runtime validation + serialization.

## Pydantic vs. TypedDict vs. dataclass — a rule of thumb, in code.

In [19]:
from pydantic import BaseModel, Field, field_validator

# Define a Movie schema with validation rules
class Movie(BaseModel):
    # Title must be a non-empty string
    title: str
    
    # Year must be greater than 1888 (the year cinema was invented)
    year: int = Field(gt=1888, description="Year must be after cinema was invented")
    
    # Rating must be between 0 and 10
    rating: float = Field(ge=0, le=10)

    # Custom validator for the title field
    @field_validator("title")
    @classmethod
    def title_not_empty(cls, v: str) -> str:
        if not v.strip():
            raise ValueError("title cannot be empty")
        return v


# ------------------ Run Code ------------------

# Valid movie
valid_movie = Movie(title="Inception", year=2010, rating=8.8)
print(valid_movie)

# Invalid movie: year too early
try:
    bad_year_movie = Movie(title="Old Film", year=1800, rating=7.0)
except Exception as e:
    print("Error (bad year):", e)

# Invalid movie: empty title
try:
    bad_title_movie = Movie(title="   ", year=2000, rating=5.0)
except Exception as e:
    print("Error (empty title):", e)

# Invalid movie: rating out of range
try:
    bad_rating_movie = Movie(title="Test", year=2000, rating=11.0)
except Exception as e:
    print("Error (bad rating):", e)


title='Inception' year=2010 rating=8.8
Error (bad year): 1 validation error for Movie
year
  Input should be greater than 1888 [type=greater_than, input_value=1800, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/greater_than
Error (empty title): 1 validation error for Movie
title
  Value error, title cannot be empty [type=value_error, input_value='   ', input_type=str]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error
Error (bad rating): 1 validation error for Movie
rating
  Input should be less than or equal to 10 [type=less_than_equal, input_value=11.0, input_type=float]
    For further information visit https://errors.pydantic.dev/2.13/v/less_than_equal


# unhappy path — validation failures and strict mode.

In [28]:
from pydantic import BaseModel, Field, field_validator
from langchain_core.messages import HumanMessage
from langchain_groq import ChatGroq

class Movie(BaseModel):
    title: str
    year: int
    rating: float = Field(ge=0, le=10)

    # Validate in Python so Groq's API doesn't reject it on the server
    @field_validator("year")
    @classmethod
    def validate_year(cls, v: int) -> int:
        if v <= 1888:
            raise ValueError("Year must be greater than 1888 (cinema was invented in 1888)")
        return v

model = ChatGroq(model="openai/gpt-oss-120b", temperature=0.7)

structured_model = model.with_structured_output(Movie, include_raw=True)

# Prompt the model to populate the Movie entry with the invalid year
response = structured_model.invoke(
    [HumanMessage(content="Generate a Movie entry for a movie set in 500 BC with release year 500 and rating 7.5.")]
)

# Gracefully handle parsing
if response["parsing_error"]:
    print("Parsing failed:", response["parsing_error"])
    print("\nRaw output from LLM:", response["raw"].tool_calls)
else:
    print("Parsed object:", response["parsed"])


Parsing failed: 1 validation error for Movie
year
  Value error, Year must be greater than 1888 (cinema was invented in 1888) [type=value_error, input_value=500, input_type=int]
    For further information visit https://errors.pydantic.dev/2.13/v/value_error

Raw output from LLM: [{'name': 'Movie', 'args': {'rating': 7.5, 'title': 'Chronicles of the Dawn', 'year': 500}, 'id': 'fc_1d0925a4-e6ba-4ecb-9628-eb464141a863', 'type': 'tool_call'}]


In [29]:
def get_structured_with_retry(model, schema, prompt, max_attempts=3):
    structured_model = model.with_structured_output(schema, include_raw=True)
    last_error = None
    for attempt in range(max_attempts):
        result = structured_model.invoke(prompt if attempt == 0 else f"{prompt}\n\nYour previous answer was invalid: {last_error}. Please correct it.")
        if result["parsing_error"] is None:
            return result["parsed"]
        last_error = result["parsing_error"]
        raise ValueError(f"Model failed to produce valid {schema.__name__} after { max_attempts} attempts")

# checking model.profile before relying on structured output.

In [30]:
def get_structured(model, schema, prompt):
    if not model.profile.get("structured_output"):
        print(f"Warning: {model} may not support native structured output; "f"with_structured_output will fall back to a tool-calling strategy.")
    return model.with_structured_output(schema).invoke(prompt)